# Patient Specific Model

In [1]:
from pathlib import Path
import numpy as np
import h5py

import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC

from sklearn.metrics import roc_auc_score, average_precision_score


In [2]:
import numpy as np
import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectPercentile, f_classif

from sklearn.svm import SVC

from sklearn.model_selection import GroupKFold, GridSearchCV

from sklearn.metrics import roc_auc_score, average_precision_score


# 1. Separate Records

> Create a dictionary with the all the data together

In [4]:

records = {}

# =====================================================
# LOAD SEIZURE FILES
# =====================================================

seizure_files = sorted(
    Path("Features/Seizure").glob("chb02_*_seizure.mat")
)

for file in seizure_files:

    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]
        y = f["all_labels"][:].squeeze()

    if X.shape[0] == 1530:
        X = X.T

    record_name = file.stem.replace("_seizure", "")

    records[record_name] = (X, y)

    print(
        "[SEIZURE]",
        record_name,
        X.shape,
        y.shape,
        "positives =", int(np.sum(y))
    )

# =====================================================
# LOAD NON-SEIZURE FILES
# =====================================================

non_files = sorted(
    Path("Features/Non_Seizure").glob("chb02_*_features.mat")
)

for file in non_files:

    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]

    if X.shape[0] == 1530:
        X = X.T

    y = np.zeros(X.shape[0], dtype=int)

    record_name = file.stem.replace("_features", "")

    records[record_name] = (X, y)

    print(
        "[NON-SEIZURE]",
        record_name,
        X.shape,
        y.shape,
        "positives =", int(np.sum(y))
    )

print("\nTotal records:", len(records))

[SEIZURE] chb02_16+ (1437, 1530) (1437,) positives = 33
[SEIZURE] chb02_16 (380, 1530) (380,) positives = 33
[SEIZURE] chb02_19 (1437, 1530) (1437,) positives = 4
[NON-SEIZURE] chb02_01 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_02 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_03 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_04 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_05 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_06 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_07 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_08 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_09 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_10 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_11 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_12 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_13 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_14 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb02_15 (1

>records = { <br>
    "chb01_03": (X_03, y_03),<br>
    "chb01_04": (X_04, y_04),<br>
    "chb01_05": (X_05, y_05),<br>
    ...<br>
}

## 1.1  Separate Seizure and Non-seizure

> Separatre the dictionary in 2

In [5]:
# =====================================================
# SPLIT SEIZURE / NON-SEIZURE RECORDS
# =====================================================
seizure_records = {
    k:v for k,v in records.items()
    if np.sum(v[1]) > 0
}

non_records = {
    k:v for k,v in records.items()
    if np.sum(v[1]) == 0
}


print("Seizure:", len(seizure_records))
print("Non seizure:", len(non_records))

Seizure: 3
Non seizure: 33


## 1.2 Pick $R$ Non-seizure Records

In [6]:
# =====================================================
# SELECT NON-SEIZURE RECORDS FOR TRAINING
# =====================================================

r = 0
random.seed(42)


selected_non = random.sample(
    list(non_records.keys()),
    r
)


print("Selected non seizure:")
print(selected_non)

Selected non seizure:
[]


# 2. Training Function

> This will remove the test record from train

In [7]:
def create_train_test(test_record):

    X_train = []
    y_train = []
    train_groups = []


    # -------------------------
    # seizure records
    # -------------------------

    for name, (X,y) in seizure_records.items():

        if name == test_record:
            continue


        X_train.append(X)
        y_train.append(y)

        train_groups.extend(
            [name] * len(y)
        )



    # -------------------------
    # non seizure records
    # -------------------------

    for name in selected_non:

        X,y = non_records[name]

        X_train.append(X)
        y_train.append(y)

        train_groups.extend(
            [name] * len(y)
        )



    # combine

    X_train = np.vstack(X_train)
    y_train = np.concatenate(y_train)

    train_groups = np.array(train_groups)



    # test

    X_test, y_test = seizure_records[test_record]


    return (
        X_train,
        y_train,
        X_test,
        y_test,
        train_groups
    )

# 3. Model

> Things that we want to tune:</br>
> - k number of features </br>
> - Value of C

In [8]:
from joblib import Memory


In [8]:
import numpy as np
import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectPercentile, f_classif

from sklearn.svm import SVC

from sklearn.model_selection import GroupKFold, RandomizedSearchCV

from sklearn.metrics import roc_auc_score, average_precision_score

from joblib import Memory
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# =====================================================
# CACHE
# =====================================================

memory = Memory(
    location="./cache",
    verbose=0
)



# =====================================================
# PIPELINE FOR TUNING
# =====================================================

pipeline = Pipeline([

    ("scale", StandardScaler()),

    ("select", SelectPercentile(
        f_classif
    )),

    ("svm", SVC(
        kernel="rbf",
        probability=False
    ))

],
memory=memory)



# =====================================================
# PARAMETERS
# =====================================================

param_grid = {


    "select__percentile":[
        2,
        5,
        10
    ],


    "svm__C":[
        0.1,
        1,
        10
    ],


    "svm__gamma":[
        "scale"
    ]

}



# =====================================================
# OUTER LORO
# =====================================================

results = {}



for test_record in seizure_records.keys():


    print("\n================")
    print("TEST:", test_record)
    print("================")



    (
        X_train,
        y_train,
        X_test,
        y_test,
        train_groups

    ) = create_train_test(test_record)



    print(
        "Train:",
        X_train.shape,
        "positives:",
        np.sum(y_train)
    )


    print(
        "Test:",
        X_test.shape,
        "positives:",
        np.sum(y_test)
    )



    # =================================================
    # INNER GROUP CV
    # =================================================

    inner_cv = GroupKFold(
        n_splits=3
    )



    search = RandomizedSearchCV(

        estimator=pipeline,

        param_distributions=param_grid,

        n_iter=9,

        scoring="average_precision",

        cv=inner_cv,

        n_jobs=-1,

        random_state=42

    )



    search.fit(

        X_train,

        y_train,

        groups=train_groups

    )



    print(
        "Best params:",
        search.best_params_
    )



    # =================================================
    # RETRAIN BEST MODEL WITH PROBABILITY
    # =================================================

    best_percentile = search.best_params_[
        "select__percentile"
    ]

    best_C = search.best_params_[
        "svm__C"
    ]

    best_gamma = search.best_params_[
        "svm__gamma"
    ]



    final_model = Pipeline([


        ("scale", StandardScaler()),


        ("select", SelectPercentile(
            f_classif,
            percentile=best_percentile
        )),


        ("svm", SVC(
            kernel="rbf",
            C=best_C,
            gamma=best_gamma,
            probability=True
        ))

    ])



    final_model.fit(
        X_train,
        y_train
    )



    # =================================================
    # TEST
    # =================================================

    y_prob = final_model.predict_proba(
        X_test
    )[:,1]


    # convert probability to class
    y_pred = (y_prob >= 0.5).astype(int)



    # confusion matrix

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()



    # metrics

    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )


    sensitivity = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )


    specificity = tn / (tn + fp)


    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )



    print("Accuracy:", accuracy)

    print("Sensitivity (Recall):", sensitivity)

    print("Specificity:", specificity)

    print("Precision:", precision)

    print("F1:", f1)



    # keep AUC/AUPRC too

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    auprc = average_precision_score(
        y_test,
        y_prob
    )


    print("AUC:", auc)

    print("AUPRC:", auprc)



print("\nDONE")


TEST: chb01_03
Train: (8112, 1530) positives: 164
Test: (1437, 1530) positives: 16
Best params: {'svm__gamma': 'scale', 'svm__C': 1, 'select__percentile': 2}
Accuracy: 0.988169798190675
Sensitivity (Recall): 0.25
Specificity: 0.9964813511611541
Precision: 0.4444444444444444
F1: 0.32
AUC: 0.9278676988036593
AUPRC: 0.3949116778930749

TEST: chb01_04
Train: (8112, 1530) positives: 169
Test: (1437, 1530) positives: 11
Best params: {'svm__gamma': 'scale', 'svm__C': 1, 'select__percentile': 2}
Accuracy: 0.9895615866388309
Sensitivity (Recall): 0.6363636363636364
Specificity: 0.9922861150070126
Precision: 0.3888888888888889
F1: 0.4827586206896552
AUC: 0.9944536529389265
AUPRC: 0.6866061803808841

TEST: chb01_15
Train: (8112, 1530) positives: 164
Test: (1437, 1530) positives: 16
Best params: {'svm__gamma': 'scale', 'svm__C': 1, 'select__percentile': 10}
Accuracy: 0.9798190675017397
Sensitivity (Recall): 0.125
Specificity: 0.9894440534834623
Precision: 0.11764705882352941
F1: 0.121212121212121

In [9]:
import numpy as np
import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectPercentile, f_classif

from sklearn.svm import SVC

from sklearn.model_selection import GroupKFold, RandomizedSearchCV

from sklearn.metrics import roc_auc_score, average_precision_score

from joblib import Memory
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# =====================================================
# CACHE
# =====================================================

memory = Memory(
    location="./cache",
    verbose=0
)



# =====================================================
# PIPELINE FOR TUNING
# =====================================================

pipeline = Pipeline([

    ("scale", StandardScaler()),

    ("select", SelectPercentile(
        f_classif
    )),

    ("svm", SVC(
        kernel="rbf",
        probability=False
    ))

],
memory=memory)



# =====================================================
# PARAMETERS
# =====================================================

param_grid = {


    "select__percentile":[
        2,
        5,
        10
    ],


    "svm__C":[
        0.1,
        1,
        10
    ],


    "svm__gamma":[
        "scale"
    ]

}



# =====================================================
# OUTER LORO
# =====================================================

results = {}



for test_record in seizure_records.keys():


    print("\n================")
    print("TEST:", test_record)
    print("================")



    (
        X_train,
        y_train,
        X_test,
        y_test,
        train_groups

    ) = create_train_test(test_record)



    print(
        "Train:",
        X_train.shape,
        "positives:",
        np.sum(y_train)
    )


    print(
        "Test:",
        X_test.shape,
        "positives:",
        np.sum(y_test)
    )



    # =================================================
    # INNER GROUP CV
    # =================================================

    inner_cv = GroupKFold(
        n_splits=3
    )



    search = GridSearchCV(

    estimator=pipeline,

    param_grid=param_grid,

    scoring="average_precision",

    cv=inner_cv,

    n_jobs=-8

)



    search.fit(

        X_train,

        y_train,

        groups=train_groups

    )



    print(
        "Best params:",
        search.best_params_
    )



    # =================================================
    # RETRAIN BEST MODEL WITH PROBABILITY
    # =================================================

    best_percentile = search.best_params_[
        "select__percentile"
    ]

    best_C = search.best_params_[
        "svm__C"
    ]

    best_gamma = search.best_params_[
        "svm__gamma"
    ]



    final_model = Pipeline([


        ("scale", StandardScaler()),


        ("select", SelectPercentile(
            f_classif,
            percentile=best_percentile
        )),


        ("svm", SVC(
            kernel="rbf",
            C=best_C,
            gamma=best_gamma,
            probability=True
        ))

    ])



    final_model.fit(
        X_train,
        y_train
    )



    # =================================================
    # TEST
    # =================================================

    y_prob = final_model.predict_proba(
        X_test
    )[:,1]


    # convert probability to class
    y_pred = (y_prob >= 0.5).astype(int)



    # confusion matrix

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()



    # metrics

    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )


    sensitivity = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )


    specificity = tn / (tn + fp)


    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )



    print("Accuracy:", accuracy)

    print("Sensitivity (Recall):", sensitivity)

    print("Specificity:", specificity)

    print("Precision:", precision)

    print("F1:", f1)



    # keep AUC/AUPRC too

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    auprc = average_precision_score(
        y_test,
        y_prob
    )


    print("AUC:", auc)

    print("AUPRC:", auprc)



print("\nDONE")


TEST: chb01_03
Train: (8112, 1530) positives: 164
Test: (1437, 1530) positives: 16
Best params: {'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 'scale'}
Accuracy: 0.988169798190675
Sensitivity (Recall): 0.25
Specificity: 0.9964813511611541
Precision: 0.4444444444444444
F1: 0.32
AUC: 0.9278676988036593
AUPRC: 0.3949116778930749

TEST: chb01_04
Train: (8112, 1530) positives: 169
Test: (1437, 1530) positives: 11


KeyboardInterrupt: 

In [13]:
import numpy as np
import pandas as pd
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0     # seconds
STEP       = 2.5      # seconds  (75% overlap)
GRID_FS    = 1        # scoring grid resolution (Hz). 1 = one label/second.
THRESHOLD  = 0.5      # operating point on y_prob (calibrate later)
K_PERSIST  = 2        # consecutive positive windows required to raise an alarm

SEIZ = pd.read_csv("seizure_annotations.csv")

# =====================================================
# STEP 1: window probabilities -> per-second hypothesis mask
# any-overlap pooling: a second is positive if ANY window covering it,
# that also passes the persistence rule, is positive.
# =====================================================
def windows_to_event_mask(y_prob, duration_sec):
    """y_prob: array, one score per window, in record order.
       returns a per-second boolean mask of length round(duration_sec*GRID_FS)."""
    pos = (y_prob >= THRESHOLD).astype(int)

    # persistence: keep only windows that are part of a run of >= K_PERSIST
    alarm_win = np.zeros_like(pos)
    run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= K_PERSIST:
            alarm_win[i - K_PERSIST + 1 : i + 1] = 1  # backfill the whole run

    n_grid = int(round(duration_sec * GRID_FS))
    mask = np.zeros(n_grid, dtype=bool)
    for i, a in enumerate(alarm_win):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n_grid)] = True
    return mask, n_grid


def ref_mask(record, n_grid):
    """ground-truth per-second mask from seizure_annotations.csv"""
    mask = np.zeros(n_grid, dtype=bool)
    rows = SEIZ[SEIZ["file"] == record]
    for _, r in rows.iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n_grid)] = True
    return mask

# =====================================================
# STEP 2: score every record, accumulate TP/FP/FN
# fold_outputs: list of (record_id, y_prob, duration_sec)
#   include ALL test records (seizure-free too) for honest FP count.
# =====================================================
def score_event_level(fold_outputs):
    TP = FP = FN = 0
    per_record = []
    for record, y_prob, duration_sec in fold_outputs:
        hyp_mask, n_grid = windows_to_event_mask(y_prob, duration_sec)
        ref = ref_mask(record, n_grid)

        ref_ann = Annotation(ref, GRID_FS)
        hyp_ann = Annotation(hyp_mask, GRID_FS)
        s = scoring.EventScoring(ref_ann, hyp_ann)   # SzCORE defaults

        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
        per_record.append((record, s.tp, s.fp, s.refTrue - s.tp, duration_sec))

    sens = TP / (TP + FN) if (TP + FN) else 0.0
    prec = TP / (TP + FP) if (TP + FP) else 0.0
    f1   = 2 * sens * prec / (sens + prec) if (sens + prec) else 0.0
    total_days = sum(d for _, _, _, _, d in per_record) / 86400
    fp_day = FP / total_days if total_days else 0.0

    print("===== EVENT-LEVEL (SzCORE) =====")
    print(f"TP={TP}  FP={FP}  FN={FN}")
    print(f"Sensitivity : {sens:.3f}")
    print(f"Precision   : {prec:.3f}")
    print(f"F1          : {f1:.3f}")
    print(f"FP/day      : {fp_day:.3f}")
    return dict(TP=TP, FP=FP, FN=FN, sens=sens, prec=prec, f1=f1, fp_day=fp_day,
                per_record=per_record)

In [14]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1
THRESHOLD  = 0.5      # operating point (calibrate later on OOF train scores)
K_PERSIST  = 2        # consecutive positive windows to raise an alarm

SEIZ = pd.read_csv("seizure_annotations.csv")

param_grid = {
    "select__percentile": [2, 5, 10],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale"],
}

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf", probability=True)),
    ])

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# STEP A: train one model per LORO fold over SEIZURE records.
# Each seizure record is held out once -> gives its test scores.
# Also fit ONE model on ALL seizure records to score the
# seizure-free records (never in training -> no leakage).
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=3)
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=-1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_

fold_outputs = []     # (record, y_prob, y_true, duration) for ALL records
seg_scores, seg_labels = [], []   # pooled out-of-fold, for segment metrics

# --- seizure records: leave-one-record-out ---
for test_record in seizure_records.keys():
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best = tune_fit(Xtr, ytr, grps)
    Xte, yte = seizure_records[test_record]
    prob = model.predict_proba(Xte)[:, 1]
    fold_outputs.append((test_record, prob, yte, duration_of(yte)))
    seg_scores.append(prob); seg_labels.append(yte)
    print(f"[SEIZ] {test_record}  best={best}  test_pos={int(yte.sum())}")

# --- seizure-free records: scored by model trained on ALL seizure records ---
Xall, yall, gall = build_train(exclude=None)
full_model, _ = tune_fit(Xall, yall, gall)
for name, (X, y) in non_records.items():
    prob = full_model.predict_proba(X)[:, 1]
    fold_outputs.append((name, prob, y, duration_of(y)))
    seg_scores.append(prob); seg_labels.append(y)
    print(f"[NON ] {name}  scored for FP")

# =====================================================
# STEP B: SEGMENT-LEVEL (pooled out-of-fold)
# =====================================================
S = np.concatenate(seg_scores)
L = np.concatenate(seg_labels)
seg_pred = (S >= THRESHOLD).astype(int)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC       : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC     : {roc_auc_score(L, S):.3f}")
print(f"F1@{THRESHOLD}     : {f1_score(L, seg_pred, zero_division=0):.3f}")
print(f"Prevalence  : {L.mean():.4f}")

# =====================================================
# STEP C: EVENT-LEVEL (SzCORE via timescoring)
# =====================================================
def windows_to_mask(y_prob, duration):
    pos = (y_prob >= THRESHOLD).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= K_PERSIST:
            alarm[i - K_PERSIST + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

TP = FP = FN = 0
for record, prob, _, dur in fold_outputs:
    hyp, n = windows_to_mask(prob, dur)
    ref = ref_mask(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp

sens = TP / (TP + FN) if (TP + FN) else 0.0
prec = TP / (TP + FP) if (TP + FP) else 0.0
f1   = 2 * sens * prec / (sens + prec) if (sens + prec) else 0.0
days = sum(d for _, _, _, d in fold_outputs) / 86400
print("\n===== EVENT-LEVEL (SzCORE) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity : {sens:.3f}")
print(f"Precision   : {prec:.3f}")
print(f"F1          : {f1:.3f}")
print(f"FP/day      : {FP/days:.3f}")

[SEIZ] chb01_03  best={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=16
[SEIZ] chb01_04  best={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=11
[SEIZ] chb01_15  best={'select__percentile': 10, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=16
[SEIZ] chb01_16  best={'select__percentile': 10, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=21
[SEIZ] chb01_18  best={'select__percentile': 10, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=37
[SEIZ] chb01_21  best={'select__percentile': 5, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=38
[SEIZ] chb01_26  best={'select__percentile': 5, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=41
[NON ] chb01_01  scored for FP
[NON ] chb01_02  scored for FP
[NON ] chb01_05  scored for FP
[NON ] chb01_06  scored for FP
[NON ] chb01_07  scored for FP
[NON ] chb01_08  scored for FP
[NON ] chb01_09  scored for FP
[NON ] chb01_10  scored for FP
[NON ] chb01_11  scored for FP
[NON ] chb01_12  scored for FP
[NON ] c

In [14]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

# calibration search grids (applied to OOF TRAIN scores only)
THRESH_GRID = None            # None -> derived per fold from score quantiles
K_GRID      = [1, 2, 3, 4, 6, 8]
FP_PER_DAY_BUDGET = 1.9       # Shoeb 0.08 FP/h = 1.92 FP/day ; a-priori ceiling

param_grid = {
    "select__percentile": [1, 2, 5, 10],
    "svm__C": [0.01,0.1, 1, 10],
    "svm__gamma": ["scale",0.01, 0.1],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf")),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    """record_scores: list of (record, scores, y, duration).
       returns total TP, FP, FN over those records."""
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

def detection_latencies(record, sc, thr, k):
    """Latency per DETECTED seizure: onset -> end of the alarm-defining window
    (the k-th consecutive positive). Undefined for misses, so only detected
    seizures contribute."""
    TOL_PRE, TOL_POST = 30, 60          # SzCORE matching tolerances
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:                                   # alarm raised exactly here
            alarm_end_times.append(i * STEP + WINDOW_LEN)   # end-time of k-th window
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in alarm_end_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)             # earliest matching alarm
    return lat

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Sweep (threshold, k); pick max sensitivity s.t. FP/day <= budget.
# Tie / infeasible -> fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        qs = np.quantile(all_scores, np.linspace(0.50, 0.999, 25))
        thr_grid = np.unique(qs)
    else:
        thr_grid = THRESH_GRID

    best = None  # (sens, -fp_day, f1, thr, k)
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            cand = (sens, -fp_day, f1, thr, k)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or cand > best_feasible:
                    best_feasible = cand
            if best is None or (f1, sens) > (best[2], best[0]):
                best = cand
    chosen = best_feasible if best_feasible is not None else best
    return chosen[3], chosen[4]   # threshold, k

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=3)
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=-1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def oof_train_record_scores(search_estimator, exclude):
    """Recompute OOF scores per TRAINING seizure record using GroupKFold,
    so calibration sees scores the model didn't train on."""
    names = [n for n in seizure_records if n != exclude]
    out = []
    for name in names:
        Xtr, ytr, grps = build_train(exclude=None)  # all seizure records
        # mask: train on all-but-this-name, score this-name
        m_test = (grps == name)
        m_train = ~m_test & (grps != exclude)
        est = make_pipe().set_params(**{k.replace("estimator__",""):v
                                        for k,v in search_estimator.get_params().items()
                                        if k in ("select__percentile","svm__C","svm__gamma")})
        est.fit(Xtr[m_train], ytr[m_train])
        sc = est.decision_function(Xtr[m_test])
        y_name = ytr[m_test]
        out.append((name, sc, y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO
# =====================================================
test_outputs = []        # (record, scores, y, dur) for ALL records (test)
seg_scores, seg_labels = [], []
fold_choices = []

for test_record in seizure_records.keys():
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, search = tune_fit(Xtr, ytr, grps)

    # --- calibrate (threshold,k) on OOF TRAIN scores, NOT on test ---
    oof = oof_train_record_scores(search.best_estimator_, exclude=test_record)
    thr, k = calibrate(oof)
    fold_choices.append((test_record, thr, k))

    # --- apply to held-out test record ---
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    test_outputs.append((test_record, sc, yte, duration_of(yte), thr, k))
    seg_scores.append(sc); seg_labels.append(yte)
    print(f"[SEIZ] {test_record}  params={best}  thr={thr:.3f} k={k}  test_pos={int(yte.sum())}")

# --- seizure-free records: scored by model on ALL seizure records ---
Xall, yall, gall = build_train(exclude=None)
full_model, _, _ = tune_fit(Xall, yall, gall)
# calibrate a single (thr,k) for FP scoring from full OOF
oof_full = oof_train_record_scores(full_model, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)

# =====================================================
# SEGMENT-LEVEL (pooled OOF, threshold-free diagnostics)
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# EVENT-LEVEL (per-record calibrated thr,k)
# =====================================================
TP = FP = FN = 0
all_latencies = []
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    all_latencies.extend(detection_latencies(record, sc, thr, k))

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

[SEIZ] chb01_03  params={'select__percentile': 2, 'svm__C': 10, 'svm__gamma': 0.01}  thr=-0.306 k=8  test_pos=16
[SEIZ] chb01_04  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 0.01}  thr=1.016 k=2  test_pos=11
[SEIZ] chb01_15  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 0.01}  thr=-0.965 k=8  test_pos=16
[SEIZ] chb01_16  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 0.01}  thr=-0.849 k=8  test_pos=21
[SEIZ] chb01_18  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 0.01}  thr=-0.638 k=8  test_pos=37
[SEIZ] chb01_21  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 0.01}  thr=-0.492 k=8  test_pos=38
[SEIZ] chb01_26  params={'select__percentile': 5, 'svm__C': 1, 'svm__gamma': 0.01}  thr=-0.628 k=8  test_pos=41

===== SEGMENT-LEVEL (pooled OOF) =====
AUPRC   : 0.444
ROC-AUC : 0.905
Prev    : 0.0035

===== EVENT-LEVEL (SzCORE, calibrated) =====
TP=6  FP=1  FN=1
Sensitivity   : 0.857
Precision     : 0.857
F1            : 0.85

In [13]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

# calibration search grids (applied to OOF TRAIN scores only)
THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 3               # clinical cap: k<=3 -> persistence <= 7.5s (~Shoeb's 6s)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 1.9       # Shoeb 0.08 FP/h = 1.92 FP/day ; a-priori ceiling

param_grid = {
    "select__percentile": [2, 5, 10],
    "svm__C": [0.01,0.1, 1, 10,100],
    "svm__gamma": ["scale", 0.01, 0.1],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf")),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    """record_scores: list of (record, scores, y, duration).
       returns total TP, FP, FN over those records."""
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Sweep (threshold, k); pick max sensitivity s.t. FP/day <= budget.
# Tie / infeasible -> fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        qs = np.quantile(all_scores, np.linspace(0.50, 0.999, 25))
        thr_grid = np.unique(qs)
    else:
        thr_grid = THRESH_GRID

    best = None  # (sens, -fp_day, f1, thr, k)
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            cand = (sens, -fp_day, f1, thr, k)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or cand > best_feasible:
                    best_feasible = cand
            if best is None or (f1, sens) > (best[2], best[0]):
                best = cand
    chosen = best_feasible if best_feasible is not None else best
    return chosen[3], chosen[4]   # threshold, k

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=3)
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=-1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def oof_train_record_scores(search_estimator, exclude):
    """Recompute OOF scores per TRAINING seizure record using GroupKFold,
    so calibration sees scores the model didn't train on."""
    names = [n for n in seizure_records if n != exclude]
    out = []
    for name in names:
        Xtr, ytr, grps = build_train(exclude=None)  # all seizure records
        # mask: train on all-but-this-name, score this-name
        m_test = (grps == name)
        m_train = ~m_test & (grps != exclude)
        est = make_pipe().set_params(**{k.replace("estimator__",""):v
                                        for k,v in search_estimator.get_params().items()
                                        if k in ("select__percentile","svm__C","svm__gamma")})
        est.fit(Xtr[m_train], ytr[m_train])
        sc = est.decision_function(Xtr[m_test])
        y_name = ytr[m_test]
        out.append((name, sc, y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO
# =====================================================
test_outputs = []        # (record, scores, y, dur) for ALL records (test)
seg_scores, seg_labels = [], []
fold_choices = []

for test_record in seizure_records.keys():
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, search = tune_fit(Xtr, ytr, grps)

    # --- calibrate (threshold,k) on OOF TRAIN scores, NOT on test ---
    oof = oof_train_record_scores(search.best_estimator_, exclude=test_record)
    thr, k = calibrate(oof)
    fold_choices.append((test_record, thr, k))

    # --- apply to held-out test record ---
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    test_outputs.append((test_record, sc, yte, duration_of(yte), thr, k))
    seg_scores.append(sc); seg_labels.append(yte)
    print(f"[SEIZ] {test_record}  params={best}  thr={thr:.3f} k={k}  test_pos={int(yte.sum())}")

# --- seizure-free records: scored by model on ALL seizure records ---
Xall, yall, gall = build_train(exclude=None)
full_model, _, _ = tune_fit(Xall, yall, gall)
# calibrate a single (thr,k) for FP scoring from full OOF
oof_full = oof_train_record_scores(full_model, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)

# =====================================================
# SEGMENT-LEVEL (pooled OOF, threshold-free diagnostics)
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# EVENT-LEVEL (per-record calibrated thr,k)
# =====================================================
# =====================================================
# LATENCY: onset -> end of alarm-defining window (k-th consecutive positive)
# Measured only for DETECTED seizures (latency is undefined for misses).
# =====================================================
def detection_latencies(record, sc, dur, thr, k):
    """For each seizure in `record`, find the first alarm-defining window whose
    end falls at/after onset and within (onset - tol_pre, offset + tol_post).
    Returns list of latencies in seconds (one per detected seizure)."""
    TOL_PRE, TOL_POST = 30, 60        # SzCORE matching tolerances
    pos = (sc >= thr).astype(int)
    # index of the k-th window that completes each run (the alarm-defining window)
    alarm_end_times = []
    run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:                              # alarm raised exactly here
            alarm_end_times.append((i * STEP) + WINDOW_LEN)   # end-time of this window
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in alarm_end_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)        # earliest matching alarm
    return lat

TP = FP = FN = 0
all_latencies = []
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    all_latencies.extend(detection_latencies(record, sc, dur, thr, k))

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

[SEIZ] chb01_03  params={'select__percentile': 2, 'svm__C': 10, 'svm__gamma': 0.01}  thr=1.580 k=1  test_pos=16


KeyboardInterrupt: 

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import average_precision_score, roc_auc_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 8               # clinical cap on persistence (latency control)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 5.0       # a-priori ceiling (looser than Shoeb; GC trades FP for latency)

param_grid = {
    "select__percentile": [2,5,10,15],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.1],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf")),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            alarm_end_times.append(i * STEP + WINDOW_LEN)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in alarm_end_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Within FP/day budget: max sensitivity, then LOWER k (latency), then F1, then low FP.
# If nothing meets budget: fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 25)))
    else:
        thr_grid = THRESH_GRID

    best = None
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=3)
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=-1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def oof_train_record_scores(best_params, exclude):
    Xtr_all, ytr_all, grps_all = build_train(exclude=None)
    keep = (grps_all != exclude)
    Xtr_all, ytr_all, grps_all = Xtr_all[keep], ytr_all[keep], grps_all[keep]
    names = [n for n in seizure_records if n != exclude]
    hp = {p: best_params[p] for p in ("select__percentile", "svm__C", "svm__gamma")}
    out = []
    for name in names:
        m_test = (grps_all == name)
        m_train = ~m_test
        est = make_pipe().set_params(**hp)
        est.fit(Xtr_all[m_train], ytr_all[m_train])
        sc = est.decision_function(Xtr_all[m_test])
        y_name = ytr_all[m_test]
        out.append((name, sc, y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO
# =====================================================
test_outputs = []
seg_scores, seg_labels = [], []
fold_choices = []

for test_record in seizure_records.keys():
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, search = tune_fit(Xtr, ytr, grps)

    oof = oof_train_record_scores(best, exclude=test_record)
    thr, k = calibrate(oof)
    fold_choices.append((test_record, thr, k))

    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    test_outputs.append((test_record, sc, yte, duration_of(yte), thr, k))
    seg_scores.append(sc); seg_labels.append(yte)
    print(f"[SEIZ] {test_record}  params={best}  thr={thr:.3f} k={k}  test_pos={int(yte.sum())}")

Xall, yall, gall = build_train(exclude=None)
full_model, full_best, _ = tune_fit(Xall, yall, gall)
oof_full = oof_train_record_scores(full_best, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
print(f"[NON-MODEL] params={full_best}  thr={thr_fp:.3f} k={k_fp}  (scores all seizure-free records)")
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)

# =====================================================
# SEGMENT-LEVEL (pooled OOF)
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# EVENT-LEVEL
# =====================================================
TP = FP = FN = 0
all_latencies = []
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    all_latencies.extend(detection_latencies(record, sc, thr, k))

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

KeyboardInterrupt: 

In [19]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (
    GroupKFold, GridSearchCV, LeaveOneGroupOut, cross_val_predict,
)
from sklearn.metrics import average_precision_score, roc_auc_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 8               # clinical cap on persistence (latency control)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 5.0       # a-priori ceiling (looser than Shoeb; GC trades FP for latency)

param_grid = {
    "select__percentile": [5,10,15,20],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.1],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf", class_weight='balanced', cache_size=8000)),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            alarm_end_times.append(i * STEP + WINDOW_LEN)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in alarm_end_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Within FP/day budget: max sensitivity, then LOWER k (latency), then F1, then low FP.
# If nothing meets budget: fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 25)))
    else:
        thr_grid = THRESH_GRID

    best = None
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=len(np.unique(groups)))
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=-1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def oof_train_record_scores(best_params, exclude):
    # Outer-training pool (all seizure records except the held-out test record)
    Xtr, ytr, grps = build_train(exclude=exclude)
    hp = {p: best_params[p] for p in ("select__percentile", "svm__C", "svm__gamma")}
    est = make_pipe().set_params(**hp)

    # One LeaveOneGroupOut pass -> OOF decision scores for every training record
    oof_scores = cross_val_predict(
        est, Xtr, ytr, groups=grps,
        cv=LeaveOneGroupOut(), method="decision_function", n_jobs=1,
    )

    out = []
    for name in np.unique(grps):
        m = (grps == name)
        y_name = ytr[m]
        out.append((name, oof_scores[m], y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO
# =====================================================
test_outputs = []
seg_scores, seg_labels = [], []
fold_choices = []

for test_record in seizure_records.keys():
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, search = tune_fit(Xtr, ytr, grps)

    oof = oof_train_record_scores(best, exclude=test_record)
    thr, k = calibrate(oof)
    fold_choices.append((test_record, thr, k))

    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    test_outputs.append((test_record, sc, yte, duration_of(yte), thr, k))
    seg_scores.append(sc); seg_labels.append(yte)
    print(f"[SEIZ] {test_record}  params={best}  thr={thr:.3f} k={k}  test_pos={int(yte.sum())}")

Xall, yall, gall = build_train(exclude=None)
full_model, full_best, _ = tune_fit(Xall, yall, gall)
oof_full = oof_train_record_scores(full_best, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
print(f"[NON-MODEL] params={full_best}  thr={thr_fp:.3f} k={k_fp}  (scores all seizure-free records)")
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)

# =====================================================
# SEGMENT-LEVEL (pooled OOF)
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# EVENT-LEVEL
# =====================================================
TP = FP = FN = 0
all_latencies = []
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    all_latencies.extend(detection_latencies(record, sc, thr, k))

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

[SEIZ] chb01_03  params={'select__percentile': 10, 'svm__C': 1, 'svm__gamma': 'scale'}  thr=-0.029 k=6  test_pos=16


KeyboardInterrupt: 

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (
    GroupKFold, GridSearchCV, LeaveOneGroupOut, cross_val_predict,
)
from sklearn.metrics import average_precision_score, roc_auc_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 8               # clinical cap on persistence (latency control)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 5.0       # a-priori ceiling (looser than Shoeb; GC trades FP for latency)

OUTER_N_JOBS = 8              # parallelism over LORO folds (inner stays at 1)

# gamma fixed to "scale" per protocol; only K (percentile) and C are tuned
param_grid = {
    "select__percentile": [5, 10, 15, 20],
    "svm__C": [0.1, 1, 10],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf", gamma="scale",
                    cache_size=8000)),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            alarm_end_times.append(i * STEP + WINDOW_LEN)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in alarm_end_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Within FP/day budget: max sensitivity, then LOWER k (latency), then F1, then low FP.
# If nothing meets budget: fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 25)))
    else:
        thr_grid = THRESH_GRID

    best = None
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=len(np.unique(groups)))
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def oof_train_record_scores(best_params, exclude):
    # Outer-training pool (all seizure records except the held-out test record)
    Xtr, ytr, grps = build_train(exclude=exclude)
    hp = {p: best_params[p] for p in ("select__percentile", "svm__C")}
    est = make_pipe().set_params(**hp)

    # One LeaveOneGroupOut pass -> OOF decision scores for every training record
    oof_scores = cross_val_predict(
        est, Xtr, ytr, groups=grps,
        cv=LeaveOneGroupOut(), method="decision_function", n_jobs=1,
    )

    out = []
    for name in np.unique(grps):
        m = (grps == name)
        y_name = ytr[m]
        out.append((name, oof_scores[m], y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO  (outer folds parallelized; inner stays single-threaded)
# =====================================================
def process_seizure_fold(test_record):
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, _ = tune_fit(Xtr, ytr, grps)
    oof = oof_train_record_scores(best, exclude=test_record)
    thr, k = calibrate(oof)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, best=best)

results = Parallel(n_jobs=OUTER_N_JOBS)(
    delayed(process_seizure_fold)(r) for r in seizure_records.keys()
)

test_outputs = []
seg_scores, seg_labels = [], []
fold_choices = []
for res in results:  # Parallel preserves input order -> deterministic
    test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                         res["thr"], res["k"]))
    seg_scores.append(res["sc"]); seg_labels.append(res["y"])
    fold_choices.append((res["record"], res["thr"], res["k"]))
    print(f"[SEIZ] {res['record']}  params={res['best']}  "
          f"thr={res['thr']:.3f} k={res['k']}  test_pos={int(res['y'].sum())}")

# Non-model: one global threshold/k for all seizure-free records
Xall, yall, gall = build_train(exclude=None)
full_model, full_best, _ = tune_fit(Xall, yall, gall)
oof_full = oof_train_record_scores(full_best, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
print(f"[NON-MODEL] params={full_best}  thr={thr_fp:.3f} k={k_fp}  (scores all seizure-free records)")
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)

# =====================================================
# SEGMENT-LEVEL (pooled OOF)
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# EVENT-LEVEL
# =====================================================
TP = FP = FN = 0
all_latencies = []
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    all_latencies.extend(detection_latencies(record, sc, thr, k))

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

[SEIZ] chb01_03  params={'select__percentile': 5, 'svm__C': 1}  thr=-0.423 k=6  test_pos=16
[SEIZ] chb01_04  params={'select__percentile': 5, 'svm__C': 1}  thr=-0.619 k=8  test_pos=11
[SEIZ] chb01_15  params={'select__percentile': 20, 'svm__C': 10}  thr=-0.190 k=4  test_pos=16
[SEIZ] chb01_16  params={'select__percentile': 10, 'svm__C': 1}  thr=-0.728 k=8  test_pos=21
[SEIZ] chb01_18  params={'select__percentile': 15, 'svm__C': 10}  thr=-0.426 k=6  test_pos=37
[SEIZ] chb01_21  params={'select__percentile': 10, 'svm__C': 1}  thr=-0.830 k=6  test_pos=38
[SEIZ] chb01_26  params={'select__percentile': 5, 'svm__C': 1}  thr=-0.665 k=6  test_pos=41
[NON-MODEL] params={'select__percentile': 5, 'svm__C': 1}  thr=-0.463 k=6  (scores all seizure-free records)

===== SEGMENT-LEVEL (pooled OOF) =====
AUPRC   : 0.365
ROC-AUC : 0.907
Prev    : 0.0035

===== EVENT-LEVEL (SzCORE, calibrated) =====
TP=6  FP=12  FN=1
Sensitivity   : 0.857
Precision     : 0.333
F1            : 0.480
FP/day        : 8.101


In [23]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (
    GroupKFold, GridSearchCV, LeaveOneGroupOut, cross_val_predict,
)
from sklearn.metrics import average_precision_score, roc_auc_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 8               # clinical cap on persistence (latency control)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 5.0       # a-priori ceiling (looser than Shoeb; GC trades FP for latency)

OUTER_N_JOBS = 8              # parallelism over LORO folds (inner stays at 1)

PATIENT = os.environ.get("PATIENT", "chb01")   # filename prefix; set per run e.g. PATIENT=chb01

GAMMA = "scale"               # fixed per protocol; only K (percentile) and C are tuned

# gamma fixed to "scale" per protocol; only K (percentile) and C are tuned
param_grid = {
    "select__percentile": [5, 10, 15, 20],
    "svm__C": [0.1, 1, 10],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf", gamma=GAMMA,
                    cache_size=8000)),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            alarm_end_times.append(i * STEP + WINDOW_LEN)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in alarm_end_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat

def _overlap(a, b):
    return a[0] < b[1] and b[0] < a[1]

def _safe_div(a, b):
    return a / b if b else float("nan")

def event_detail(record, sc, thr, k, s):
    # Detected-seizure rows + false-alarm rows, derived from the scorer's own
    # events so counts tie out: sum(detected)==s.tp, len(false_alarms)==s.fp.
    ref_ev, hyp_ev = s.ref.events, s.hyp.events
    TOL_PRE, TOL_POST = 30, 60
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            alarm_end_times.append(i * STEP + WINDOW_LEN)

    seiz_rows = []
    for sid, (rs, re) in enumerate(ref_ev):
        detected = any(_overlap((rs, re), h) for h in hyp_ev)
        dt = lat = np.nan
        if detected:
            cands = [t for t in alarm_end_times if (rs - TOL_PRE) <= t <= (re + TOL_POST)]
            if cands:
                dt = min(cands); lat = dt - rs
        seiz_rows.append(dict(record=record, seizure_id=sid,
                              onset=float(rs), offset=float(re),
                              detected=bool(detected),
                              detection_time=dt, latency=lat))

    fa_rows, aid = [], 0
    for hs, he in hyp_ev:
        if not any(_overlap((hs, he), r) for r in ref_ev):
            fa_rows.append(dict(record=record, alarm_id=aid,
                                start_sec=float(hs), end_sec=float(he),
                                duration_sec=float(he - hs)))
            aid += 1
    return seiz_rows, fa_rows

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Within FP/day budget: max sensitivity, then LOWER k (latency), then F1, then low FP.
# If nothing meets budget: fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 25)))
    else:
        thr_grid = THRESH_GRID

    best = None
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=len(np.unique(groups)))
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def n_selected_of(model):
    return int(model.named_steps["select"].get_support().sum())

def oof_train_record_scores(best_params, exclude):
    # Outer-training pool (all seizure records except the held-out test record)
    Xtr, ytr, grps = build_train(exclude=exclude)
    hp = {p: best_params[p] for p in ("select__percentile", "svm__C")}
    est = make_pipe().set_params(**hp)

    # One LeaveOneGroupOut pass -> OOF decision scores for every training record
    oof_scores = cross_val_predict(
        est, Xtr, ytr, groups=grps,
        cv=LeaveOneGroupOut(), method="decision_function", n_jobs=1,
    )

    out = []
    for name in np.unique(grps):
        m = (grps == name)
        y_name = ytr[m]
        out.append((name, oof_scores[m], y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO  (outer folds parallelized; inner stays single-threaded)
# =====================================================
def process_seizure_fold(test_record):
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, _ = tune_fit(Xtr, ytr, grps)
    oof = oof_train_record_scores(best, exclude=test_record)
    thr, k = calibrate(oof)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, best=best, n_selected=n_selected_of(model))

results = Parallel(n_jobs=OUTER_N_JOBS)(
    delayed(process_seizure_fold)(r) for r in seizure_records.keys()
)

test_outputs = []
seg_scores, seg_labels = [], []
fold_choices = []
meta = {}   # per-record: type, C, gamma, selected_features
for res in results:  # Parallel preserves input order -> deterministic
    test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                         res["thr"], res["k"]))
    seg_scores.append(res["sc"]); seg_labels.append(res["y"])
    fold_choices.append((res["record"], res["thr"], res["k"]))
    meta[res["record"]] = dict(type="seizure", C=res["best"]["svm__C"],
                               gamma=GAMMA, selected_features=res["n_selected"])
    print(f"[SEIZ] {res['record']}  params={res['best']}  "
          f"thr={res['thr']:.3f} k={res['k']}  test_pos={int(res['y'].sum())}")

# Non-model: one global threshold/k for all seizure-free records
Xall, yall, gall = build_train(exclude=None)
full_model, full_best, _ = tune_fit(Xall, yall, gall)
full_n_selected = n_selected_of(full_model)
oof_full = oof_train_record_scores(full_best, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
print(f"[NON-MODEL] params={full_best}  thr={thr_fp:.3f} k={k_fp}  (scores all seizure-free records)")
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)
    meta[name] = dict(type="nonseizure", C=full_best["svm__C"],
                      gamma=GAMMA, selected_features=full_n_selected)

# =====================================================
# SEGMENT-LEVEL (pooled OOF)
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# EVENT-LEVEL  (+ collect per-record / per-event detail)
# =====================================================
TP = FP = FN = 0
all_latencies = []
per_record_rows = []
seizure_rows = []
false_alarm_rows = []
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    r_tp, r_fp, r_fn = s.tp, s.fp, s.refTrue - s.tp
    TP += r_tp; FP += r_fp; FN += r_fn

    lats = detection_latencies(record, sc, thr, k)
    all_latencies.extend(lats)

    seiz_rows, fa_rows = event_detail(record, sc, thr, k, s)
    seizure_rows.extend(seiz_rows)
    false_alarm_rows.extend(fa_rows)

    r_sens = _safe_div(r_tp, r_tp + r_fn)
    r_prec = _safe_div(r_tp, r_tp + r_fp)
    r_f1   = (_safe_div(2 * r_sens * r_prec, r_sens + r_prec)
              if (r_sens == r_sens and r_prec == r_prec) else float("nan"))
    r_fp_day = _safe_div(r_fp, dur / 86400)
    m = meta[record]
    per_record_rows.append({
        "record": record, "type": m["type"],
        "TP": r_tp, "FP": r_fp, "FN": r_fn,
        "sensitivity": r_sens, "precision": r_prec, "F1": r_f1,
        "FP/day": r_fp_day,
        "latency_median": float(np.median(lats)) if lats else np.nan,
        "threshold": thr, "k": k,
        "C": m["C"], "gamma": m["gamma"],
        "selected_features": m["selected_features"],
    })

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

# =====================================================
# SAVE  (3 CSVs)
# =====================================================
per_record_cols = ["record", "type", "TP", "FP", "FN", "sensitivity",
                   "precision", "F1", "FP/day", "latency_median",
                   "threshold", "k", "C", "gamma", "selected_features"]
pd.DataFrame(per_record_rows, columns=per_record_cols) \
  .to_csv(f"{PATIENT}_per_record.csv", index=False)

fa_cols = ["record", "alarm_id", "start_sec", "end_sec", "duration_sec"]
pd.DataFrame(false_alarm_rows, columns=fa_cols) \
  .to_csv(f"{PATIENT}_false_alarms.csv", index=False)

seiz_cols = ["record", "seizure_id", "onset", "offset", "detected",
             "detection_time", "latency"]
pd.DataFrame(seizure_rows, columns=seiz_cols) \
  .to_csv(f"{PATIENT}_detected_seizures.csv", index=False)

print(f"\nSaved: {PATIENT}_per_record.csv, {PATIENT}_false_alarms.csv, "
      f"{PATIENT}_detected_seizures.csv")

[SEIZ] chb01_03  params={'select__percentile': 5, 'svm__C': 1}  thr=-0.423 k=6  test_pos=16
[SEIZ] chb01_04  params={'select__percentile': 5, 'svm__C': 1}  thr=-0.619 k=8  test_pos=11
[SEIZ] chb01_15  params={'select__percentile': 20, 'svm__C': 10}  thr=-0.190 k=4  test_pos=16
[SEIZ] chb01_16  params={'select__percentile': 10, 'svm__C': 1}  thr=-0.728 k=8  test_pos=21
[SEIZ] chb01_18  params={'select__percentile': 15, 'svm__C': 10}  thr=-0.426 k=6  test_pos=37
[SEIZ] chb01_21  params={'select__percentile': 10, 'svm__C': 1}  thr=-0.830 k=6  test_pos=38
[SEIZ] chb01_26  params={'select__percentile': 5, 'svm__C': 1}  thr=-0.665 k=6  test_pos=41
[NON-MODEL] params={'select__percentile': 5, 'svm__C': 1}  thr=-0.463 k=6  (scores all seizure-free records)

===== SEGMENT-LEVEL (pooled OOF) =====
AUPRC   : 0.365
ROC-AUC : 0.907
Prev    : 0.0035

===== EVENT-LEVEL (SzCORE, calibrated) =====
TP=6  FP=12  FN=1
Sensitivity   : 0.857
Precision     : 0.333
F1            : 0.480
FP/day        : 8.101


In [25]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (
    GroupKFold, GridSearchCV, LeaveOneGroupOut, cross_val_predict,
)
from sklearn.metrics import average_precision_score, roc_auc_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 8               # clinical cap on persistence (latency control)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 5.0       # a-priori ceiling (looser than Shoeb; GC trades FP for latency)

OUTER_N_JOBS = 8              # parallelism over LORO folds (inner stays at 1)

PATIENT = os.environ.get("PATIENT", "chbXX")   # filename prefix; set per run e.g. PATIENT=chb01

# gamma fixed to "scale" per protocol; only K (percentile) and C are tuned
param_grid = {
    "select__percentile": [5, 10, 15, 20],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.1, 0.01],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf", gamma=GAMMA,
                    cache_size=8000)),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            alarm_end_times.append(i * STEP + WINDOW_LEN)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in alarm_end_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat

def _overlap(a, b):
    return a[0] < b[1] and b[0] < a[1]

def _safe_div(a, b):
    return a / b if b else float("nan")

def event_detail(record, sc, thr, k, s):
    # Detected-seizure rows + false-alarm rows, derived from the scorer's own
    # events so counts tie out: sum(detected)==s.tp, len(false_alarms)==s.fp.
    ref_ev, hyp_ev = s.ref.events, s.hyp.events
    TOL_PRE, TOL_POST = 30, 60
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            alarm_end_times.append(i * STEP + WINDOW_LEN)

    seiz_rows = []
    for sid, (rs, re) in enumerate(ref_ev):
        detected = any(_overlap((rs, re), h) for h in hyp_ev)
        dt = lat = np.nan
        if detected:
            cands = [t for t in alarm_end_times if (rs - TOL_PRE) <= t <= (re + TOL_POST)]
            if cands:
                dt = min(cands); lat = dt - rs
        seiz_rows.append(dict(record=record, seizure_id=sid,
                              onset=float(rs), offset=float(re),
                              detected=bool(detected),
                              detection_time=dt, latency=lat))

    fa_rows, aid = [], 0
    for hs, he in hyp_ev:
        if not any(_overlap((hs, he), r) for r in ref_ev):
            fa_rows.append(dict(record=record, alarm_id=aid,
                                start_sec=float(hs), end_sec=float(he),
                                duration_sec=float(he - hs)))
            aid += 1
    return seiz_rows, fa_rows

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Within FP/day budget: max sensitivity, then LOWER k (latency), then F1, then low FP.
# If nothing meets budget: fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 25)))
    else:
        thr_grid = THRESH_GRID

    best = None
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=len(np.unique(groups)))
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def n_selected_of(model):
    return int(model.named_steps["select"].get_support().sum())

def oof_train_record_scores(best_params, exclude):
    # Outer-training pool (all seizure records except the held-out test record)
    Xtr, ytr, grps = build_train(exclude=exclude)
    hp = {p: best_params[p] for p in ("select__percentile", "svm__C","svm__gamma")}
    est = make_pipe().set_params(**hp)

    # One LeaveOneGroupOut pass -> OOF decision scores for every training record
    oof_scores = cross_val_predict(
        est, Xtr, ytr, groups=grps,
        cv=LeaveOneGroupOut(), method="decision_function", n_jobs=1,
    )

    out = []
    for name in np.unique(grps):
        m = (grps == name)
        y_name = ytr[m]
        out.append((name, oof_scores[m], y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO  (outer folds parallelized; inner stays single-threaded)
# =====================================================
def process_seizure_fold(test_record):
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, _ = tune_fit(Xtr, ytr, grps)
    oof = oof_train_record_scores(best, exclude=test_record)
    thr, k = calibrate(oof)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, best=best, n_selected=n_selected_of(model))

results = Parallel(n_jobs=OUTER_N_JOBS)(
    delayed(process_seizure_fold)(r) for r in seizure_records.keys()
)

test_outputs = []
seg_scores, seg_labels = [], []
fold_choices = []
meta = {}   # per-record: type, C, gamma, selected_features
for res in results:  # Parallel preserves input order -> deterministic
    test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                         res["thr"], res["k"]))
    seg_scores.append(res["sc"]); seg_labels.append(res["y"])
    fold_choices.append((res["record"], res["thr"], res["k"]))
    meta[res["record"]] = dict(type="seizure", C=res["best"]["svm__C"],
                               gamma=GAMMA, selected_features=res["n_selected"])
    print(f"[SEIZ] {res['record']}  params={res['best']}  "
          f"thr={res['thr']:.3f} k={res['k']}  test_pos={int(res['y'].sum())}")

# Non-model: one global threshold/k for all seizure-free records
Xall, yall, gall = build_train(exclude=None)
full_model, full_best, _ = tune_fit(Xall, yall, gall)
full_n_selected = n_selected_of(full_model)
oof_full = oof_train_record_scores(full_best, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
print(f"[NON-MODEL] params={full_best}  thr={thr_fp:.3f} k={k_fp}  (scores all seizure-free records)")
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)
    meta[name] = dict(type="nonseizure", C=full_best["svm__C"],
                      gamma=GAMMA, selected_features=full_n_selected)

# =====================================================
# SEGMENT-LEVEL (pooled OOF)
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# EVENT-LEVEL  (+ collect per-record / per-event detail)
# =====================================================
TP = FP = FN = 0
all_latencies = []
per_record_rows = []
seizure_rows = []
false_alarm_rows = []
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    r_tp, r_fp, r_fn = s.tp, s.fp, s.refTrue - s.tp
    TP += r_tp; FP += r_fp; FN += r_fn

    lats = detection_latencies(record, sc, thr, k)
    all_latencies.extend(lats)

    seiz_rows, fa_rows = event_detail(record, sc, thr, k, s)
    seizure_rows.extend(seiz_rows)
    false_alarm_rows.extend(fa_rows)

    r_sens = _safe_div(r_tp, r_tp + r_fn)
    r_prec = _safe_div(r_tp, r_tp + r_fp)
    r_f1   = (_safe_div(2 * r_sens * r_prec, r_sens + r_prec)
              if (r_sens == r_sens and r_prec == r_prec) else float("nan"))
    r_fp_day = _safe_div(r_fp, dur / 86400)
    m = meta[record]
    per_record_rows.append({
        "record": record, "type": m["type"],
        "TP": r_tp, "FP": r_fp, "FN": r_fn,
        "sensitivity": r_sens, "precision": r_prec, "F1": r_f1,
        "FP/day": r_fp_day,
        "record_length_sec": dur,
        "latency_median": float(np.median(lats)) if lats else np.nan,
        "threshold": thr, "k": k,
        "C": m["C"], "gamma": m["gamma"],
        "selected_features": m["selected_features"],
    })

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

# =====================================================
# SAVE  (3 CSVs)
# =====================================================
per_record_cols = ["record", "type", "TP", "FP", "FN", "sensitivity",
                   "precision", "F1", "FP/day", "record_length_sec",
                   "latency_median", "threshold", "k", "C", "gamma",
                   "selected_features"]
pd.DataFrame(per_record_rows, columns=per_record_cols) \
  .to_csv(f"{PATIENT}_per_record.csv", index=False)

fa_cols = ["record", "alarm_id", "start_sec", "end_sec", "duration_sec"]
pd.DataFrame(false_alarm_rows, columns=fa_cols) \
  .to_csv(f"{PATIENT}_false_alarms.csv", index=False)

seiz_cols = ["record", "seizure_id", "onset", "offset", "detected",
             "detection_time", "latency"]
pd.DataFrame(seizure_rows, columns=seiz_cols) \
  .to_csv(f"{PATIENT}_detected_seizures.csv", index=False)

# Master summary -> APPEND one row per patient (builds the 24-patient table)
summary = {
    "patient": PATIENT,
    "auprc": average_precision_score(L, S),
    "roc_auc": roc_auc_score(L, S),
    "prev": L.mean(),
    "TP": TP, "FP": FP, "FN": FN,
    "sensitivity": sens, "precision": prec, "F1": f1,
    "FP/day": FP / days,
    "latency_median": med_lat,
    "n_detected": len(all_latencies),
}
master = "results_summary.csv"
pd.DataFrame([summary]).to_csv(
    master, mode="a", header=not os.path.exists(master), index=False,
)

print(f"\nSaved: {PATIENT}_per_record.csv, {PATIENT}_false_alarms.csv, "
      f"{PATIENT}_detected_seizures.csv, appended row to {master}")

KeyboardInterrupt: 

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (
    GroupKFold, GridSearchCV, LeaveOneGroupOut, cross_val_predict,
)
from sklearn.metrics import average_precision_score, roc_auc_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 8               # clinical cap on persistence (latency control)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 5.0       # a-priori ceiling (looser than Shoeb; GC trades FP for latency)

OUTER_N_JOBS = 8              # parallelism over LORO folds (inner stays at 1)

PATIENT = os.environ.get("PATIENT", "chbXX")   # filename prefix; set per run e.g. PATIENT=chb01

# gamma fixed to "scale" per protocol; only K (percentile) and C are tuned
param_grid = {
    "select__percentile": [5, 10, 15, 20],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.1, 0.01],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf", gamma=GAMMA,
                    cache_size=8000)),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            alarm_end_times.append(i * STEP + WINDOW_LEN)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in alarm_end_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat

def _overlap(a, b):
    return a[0] < b[1] and b[0] < a[1]

def _safe_div(a, b):
    return a / b if b else float("nan")

# --- NEW: guarded per-record segment-level metrics ---
# AUPRC/ROC-AUC require both classes present; non-seizure records (all y=0)
# or any record with a single class would otherwise raise. Return NaN instead.
def safe_segment_metrics(y, sc):
    if len(np.unique(y)) < 2:
        return float("nan"), float("nan")
    return average_precision_score(y, sc), roc_auc_score(y, sc)

def event_detail(record, sc, thr, k, s):
    # Detected-seizure rows + false-alarm rows, derived from the scorer's own
    # events so counts tie out: sum(detected)==s.tp, len(false_alarms)==s.fp.
    ref_ev, hyp_ev = s.ref.events, s.hyp.events
    TOL_PRE, TOL_POST = 30, 60
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            alarm_end_times.append(i * STEP + WINDOW_LEN)

    seiz_rows = []
    for sid, (rs, re) in enumerate(ref_ev):
        detected = any(_overlap((rs, re), h) for h in hyp_ev)
        dt = lat = np.nan
        if detected:
            cands = [t for t in alarm_end_times if (rs - TOL_PRE) <= t <= (re + TOL_POST)]
            if cands:
                dt = min(cands); lat = dt - rs
        seiz_rows.append(dict(record=record, seizure_id=sid,
                              onset=float(rs), offset=float(re),
                              detected=bool(detected),
                              detection_time=dt, latency=lat))

    fa_rows, aid = [], 0
    for hs, he in hyp_ev:
        if not any(_overlap((hs, he), r) for r in ref_ev):
            fa_rows.append(dict(record=record, alarm_id=aid,
                                start_sec=float(hs), end_sec=float(he),
                                duration_sec=float(he - hs)))
            aid += 1
    return seiz_rows, fa_rows

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Within FP/day budget: max sensitivity, then LOWER k (latency), then F1, then low FP.
# If nothing meets budget: fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 25)))
    else:
        thr_grid = THRESH_GRID

    best = None
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=len(np.unique(groups)))
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def n_selected_of(model):
    return int(model.named_steps["select"].get_support().sum())

def oof_train_record_scores(best_params, exclude):
    # Outer-training pool (all seizure records except the held-out test record)
    Xtr, ytr, grps = build_train(exclude=exclude)
    hp = {p: best_params[p] for p in ("select__percentile", "svm__C","svm__gamma")}
    est = make_pipe().set_params(**hp)

    # One LeaveOneGroupOut pass -> OOF decision scores for every training record
    oof_scores = cross_val_predict(
        est, Xtr, ytr, groups=grps,
        cv=LeaveOneGroupOut(), method="decision_function", n_jobs=1,
    )

    out = []
    for name in np.unique(grps):
        m = (grps == name)
        y_name = ytr[m]
        out.append((name, oof_scores[m], y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO  (outer folds parallelized; inner stays single-threaded)
# =====================================================
def process_seizure_fold(test_record):
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, _ = tune_fit(Xtr, ytr, grps)
    oof = oof_train_record_scores(best, exclude=test_record)
    thr, k = calibrate(oof)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, best=best, n_selected=n_selected_of(model))

results = Parallel(n_jobs=OUTER_N_JOBS)(
    delayed(process_seizure_fold)(r) for r in seizure_records.keys()
)

test_outputs = []
seg_scores, seg_labels = [], []
fold_choices = []
meta = {}   # per-record: type, C, gamma, selected_features
for res in results:  # Parallel preserves input order -> deterministic
    test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                         res["thr"], res["k"]))
    seg_scores.append(res["sc"]); seg_labels.append(res["y"])
    fold_choices.append((res["record"], res["thr"], res["k"]))
    meta[res["record"]] = dict(type="seizure", C=res["best"]["svm__C"],
                               gamma=GAMMA, selected_features=res["n_selected"])
    print(f"[SEIZ] {res['record']}  params={res['best']}  "
          f"thr={res['thr']:.3f} k={res['k']}  test_pos={int(res['y'].sum())}")

# Non-model: one global threshold/k for all seizure-free records
Xall, yall, gall = build_train(exclude=None)
full_model, full_best, _ = tune_fit(Xall, yall, gall)
full_n_selected = n_selected_of(full_model)
oof_full = oof_train_record_scores(full_best, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
print(f"[NON-MODEL] params={full_best}  thr={thr_fp:.3f} k={k_fp}  (scores all seizure-free records)")
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)
    meta[name] = dict(type="nonseizure", C=full_best["svm__C"],
                      gamma=GAMMA, selected_features=full_n_selected)

# =====================================================
# SEGMENT-LEVEL (pooled OOF, all records combined)
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# EVENT-LEVEL  (+ per-record segment-level, + per-record/per-event detail)
# =====================================================
TP = FP = FN = 0
all_latencies = []
per_record_rows = []
seizure_rows = []
false_alarm_rows = []
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    r_tp, r_fp, r_fn = s.tp, s.fp, s.refTrue - s.tp
    TP += r_tp; FP += r_fp; FN += r_fn

    lats = detection_latencies(record, sc, thr, k)
    all_latencies.extend(lats)

    seiz_rows, fa_rows = event_detail(record, sc, thr, k, s)
    seizure_rows.extend(seiz_rows)
    false_alarm_rows.extend(fa_rows)

    r_sens = _safe_div(r_tp, r_tp + r_fn)
    r_prec = _safe_div(r_tp, r_tp + r_fp)
    r_f1   = (_safe_div(2 * r_sens * r_prec, r_sens + r_prec)
              if (r_sens == r_sens and r_prec == r_prec) else float("nan"))

    # --- NEW: per-record segment-level metrics (guarded for single-class records) ---
    seg_auprc, seg_rocauc = safe_segment_metrics(y, sc)

    m = meta[record]
    per_record_rows.append({
        "record": record, "type": m["type"],
        "record_length_sec": dur,
        # --- segment-level ---
        "segment_auprc": seg_auprc,
        "segment_rocauc": seg_rocauc,
        # --- event-level (renamed from TP/FP/FN/sensitivity/precision/F1
        #     to make explicit these are event-level, not segment-level) ---
        "event_TP": r_tp, "event_FP": r_fp, "event_FN": r_fn,
        "event_sensitivity": r_sens, "event_precision": r_prec, "event_F1": r_f1,
        "latency_median": float(np.median(lats)) if lats else np.nan,
        "threshold": thr, "k": k,
        "C": m["C"], "gamma": m["gamma"],
        "selected_features": m["selected_features"],
    })

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

# =====================================================
# SAVE  (3 CSVs — false_alarms and detected_seizures kept separate, as before)
# =====================================================
per_record_cols = ["record", "type", "record_length_sec",
                   "segment_auprc", "segment_rocauc",
                   "event_TP", "event_FP", "event_FN",
                   "event_sensitivity", "event_precision", "event_F1",
                   "latency_median", "threshold", "k", "C", "gamma",
                   "selected_features"]
pd.DataFrame(per_record_rows, columns=per_record_cols) \
  .to_csv(f"{PATIENT}_per_record.csv", index=False)

fa_cols = ["record", "alarm_id", "start_sec", "end_sec", "duration_sec"]
pd.DataFrame(false_alarm_rows, columns=fa_cols) \
  .to_csv(f"{PATIENT}_false_alarms.csv", index=False)

seiz_cols = ["record", "seizure_id", "onset", "offset", "detected",
             "detection_time", "latency"]
pd.DataFrame(seizure_rows, columns=seiz_cols) \
  .to_csv(f"{PATIENT}_detected_seizures.csv", index=False)

# Master summary -> APPEND one row per patient (builds the 24-patient table)
summary = {
    "patient": PATIENT,
    "auprc": average_precision_score(L, S),
    "roc_auc": roc_auc_score(L, S),
    "prev": L.mean(),
    "TP": TP, "FP": FP, "FN": FN,
    "sensitivity": sens, "precision": prec, "F1": f1,
    "FP/day": FP / days,
    "latency_median": med_lat,
    "n_detected": len(all_latencies),
}
master = "results_summary.csv"
pd.DataFrame([summary]).to_csv(
    master, mode="a", header=not os.path.exists(master), index=False,
)

print(f"\nSaved: {PATIENT}_per_record.csv, {PATIENT}_false_alarms.csv, "
      f"{PATIENT}_detected_seizures.csv, appended row to {master}")

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (
    GroupKFold, GridSearchCV, LeaveOneGroupOut, cross_val_predict,
)
from sklearn.metrics import average_precision_score, roc_auc_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 8               # clinical cap on persistence (latency control)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 5.0       # a-priori ceiling (looser than Shoeb; GC trades FP for latency)

OUTER_N_JOBS = 8              # parallelism over LORO folds (inner stays at 1)

PATIENT = os.environ.get("PATIENT", "chbXX")   # filename prefix; set per run e.g. PATIENT=chb01

# percentile (feature selection), C, and gamma are all tuned via the grid below
param_grid = {
    "select__percentile": [5, 10],
    "svm__C": [0.1, 1],
    "svm__gamma": ["scale", 0.01],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf",
                    cache_size=8000, class_weight='balanced')),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            alarm_end_times.append(i * STEP + WINDOW_LEN)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in alarm_end_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat

def _overlap(a, b):
    return a[0] < b[1] and b[0] < a[1]

def _safe_div(a, b):
    return a / b if b else float("nan")

# --- SEGMENT-LEVEL (window-level) confusion-matrix metrics at a given threshold ---
# Thresholds the raw window scores (no k-persistence; that is event-level logic).
# Called with thr=0.0 (the natural decision_function boundary) for standard,
# literature-comparable segment-level reporting.
# Returns accuracy, specificity, sensitivity, precision, f1.
# Undefined metrics (e.g. sensitivity on an all-negative record) -> NaN via _safe_div.
def segment_metrics(y, sc, thr):
    y = np.asarray(y).astype(int)
    pred = (sc >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    acc  = _safe_div(tp + tn, tp + tn + fp + fn)
    spec = _safe_div(tn, tn + fp)
    sens = _safe_div(tp, tp + fn)
    prec = _safe_div(tp, tp + fp)
    f1 = (_safe_div(2 * prec * sens, prec + sens)
          if (prec == prec and sens == sens) else float("nan"))
    return acc, spec, sens, prec, f1

def false_alarm_rows_of(record, s):
    # Event-level false alarms (hypothesis events not overlapping any reference event).
    ref_ev, hyp_ev = s.ref.events, s.hyp.events
    rows, aid = [], 0
    for hs, he in hyp_ev:
        if not any(_overlap((hs, he), r) for r in ref_ev):
            rows.append(dict(record=record, alarm_id=aid,
                             start_sec=float(hs), end_sec=float(he),
                             duration_sec=float(he - hs)))
            aid += 1
    return rows

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Within FP/day budget: max sensitivity, then LOWER k (latency), then F1, then low FP.
# If nothing meets budget: fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 25)))
    else:
        thr_grid = THRESH_GRID

    best = None
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=len(np.unique(groups)))
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def n_selected_of(model):
    return int(model.named_steps["select"].get_support().sum())

def oof_train_record_scores(best_params, exclude):
    # Outer-training pool (all seizure records except the held-out test record)
    Xtr, ytr, grps = build_train(exclude=exclude)
    hp = {p: best_params[p] for p in ("select__percentile", "svm__C","svm__gamma")}
    est = make_pipe().set_params(**hp)

    # One LeaveOneGroupOut pass -> OOF decision scores for every training record
    oof_scores = cross_val_predict(
        est, Xtr, ytr, groups=grps,
        cv=LeaveOneGroupOut(), method="decision_function", n_jobs=1,
    )

    out = []
    for name in np.unique(grps):
        m = (grps == name)
        y_name = ytr[m]
        out.append((name, oof_scores[m], y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO  (outer folds parallelized; inner stays single-threaded)
# =====================================================
def process_seizure_fold(test_record):
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, _ = tune_fit(Xtr, ytr, grps)
    oof = oof_train_record_scores(best, exclude=test_record)
    thr, k = calibrate(oof)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, best=best, n_selected=n_selected_of(model))

results = Parallel(n_jobs=OUTER_N_JOBS)(
    delayed(process_seizure_fold)(r) for r in seizure_records.keys()
)

test_outputs = []
seg_scores, seg_labels = [], []
fold_choices = []
meta = {}   # per-record: type, C, gamma, selected_features
for res in results:  # Parallel preserves input order -> deterministic
    test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                         res["thr"], res["k"]))
    seg_scores.append(res["sc"]); seg_labels.append(res["y"])
    fold_choices.append((res["record"], res["thr"], res["k"]))
    meta[res["record"]] = dict(type="seizure", C=res["best"]["svm__C"],
                               gamma=res["best"]["svm__gamma"],
                               selected_features=res["n_selected"])
    print(f"[SEIZ] {res['record']}  params={res['best']}  "
          f"thr={res['thr']:.3f} k={res['k']}  test_pos={int(res['y'].sum())}")

# Non-model: one global threshold/k for all seizure-free records
Xall, yall, gall = build_train(exclude=None)
full_model, full_best, _ = tune_fit(Xall, yall, gall)
full_n_selected = n_selected_of(full_model)
oof_full = oof_train_record_scores(full_best, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
print(f"[NON-MODEL] params={full_best}  thr={thr_fp:.3f} k={k_fp}  (scores all seizure-free records)")
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)
    meta[name] = dict(type="nonseizure", C=full_best["svm__C"],
                      gamma=full_best["svm__gamma"],
                      selected_features=full_n_selected)

# =====================================================
# POOLED SEGMENT-LEVEL (all records combined) — threshold-free summary
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# PER-RECORD: segment-level (thresholded) + event-level + false-alarm log
# =====================================================
TP = FP = FN = 0
all_latencies = []
per_record_rows = []
false_alarm_rows = []
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    r_tp, r_fp, r_fn = s.tp, s.fp, s.refTrue - s.tp
    TP += r_tp; FP += r_fp; FN += r_fn

    # event-level metrics for this record
    ev_sens = _safe_div(r_tp, r_tp + r_fn)
    ev_prec = _safe_div(r_tp, r_tp + r_fp)
    ev_f1   = (_safe_div(2 * ev_sens * ev_prec, ev_sens + ev_prec)
               if (ev_sens == ev_sens and ev_prec == ev_prec) else float("nan"))

    lats = detection_latencies(record, sc, thr, k)
    all_latencies.extend(lats)

    # event-level false-alarm locations
    false_alarm_rows.extend(false_alarm_rows_of(record, s))

    # segment-level (window-level) metrics at the natural decision boundary
    # (decision_function >= 0), i.e. the classifier's own predict() boundary.
    # Kept independent of the event-level calibrated threshold so these numbers
    # are comparable to standard segment-level reporting in the literature.
    seg_acc, seg_spec, seg_sens, seg_prec, seg_f1 = segment_metrics(y, sc, 0.0)

    m = meta[record]
    per_record_rows.append({
        "patient": PATIENT,
        "record": record,
        "n_windows": int(len(y)),
        "n_seizure_windows": int(np.asarray(y).sum()),
        "record_length_sec": dur,
        # --- parameters ---
        "C": m["C"], "gamma": m["gamma"],
        "selected_features": m["selected_features"],
        "threshold": thr, "k": k,
        # --- segment-level (window-level, thresholded) ---
        "seg_accuracy": seg_acc,
        "seg_specificity": seg_spec,
        "seg_sensitivity": seg_sens,
        "seg_precision": seg_prec,
        "seg_f1": seg_f1,
        # --- event-level ---
        "event_sensitivity": ev_sens,
        "event_precision": ev_prec,
        "event_f1": ev_f1,
        "event_n_false_alarms": int(r_fp),
        "latency_median": float(np.median(lats)) if lats else np.nan,
    })

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

# =====================================================
# SAVE  (2 CSVs)
#   1) per-record metrics    2) event-level false-alarm locations
# =====================================================
per_record_cols = ["patient", "record", "n_windows", "n_seizure_windows",
                   "record_length_sec",
                   "C", "gamma", "selected_features", "threshold", "k",
                   "seg_accuracy", "seg_specificity", "seg_sensitivity",
                   "seg_precision", "seg_f1",
                   "event_sensitivity", "event_precision", "event_f1",
                   "event_n_false_alarms", "latency_median"]
pd.DataFrame(per_record_rows, columns=per_record_cols) \
  .to_csv(f"{PATIENT}_per_record.csv", index=False)

fa_cols = ["record", "alarm_id", "start_sec", "end_sec", "duration_sec"]
pd.DataFrame(false_alarm_rows, columns=fa_cols) \
  .to_csv(f"{PATIENT}_false_alarms.csv", index=False)

# Master summary -> APPEND one row per patient (builds the 24-patient table)
summary = {
    "patient": PATIENT,
    "auprc": average_precision_score(L, S),
    "roc_auc": roc_auc_score(L, S),
    "prev": L.mean(),
    "TP": TP, "FP": FP, "FN": FN,
    "sensitivity": sens, "precision": prec, "f1": f1,
    "FP/day": FP / days,
    "latency_median": med_lat,
    "n_detected": len(all_latencies),
}
master = "results_summary.csv"
pd.DataFrame([summary]).to_csv(
    master, mode="a", header=not os.path.exists(master), index=False,
)

print(f"\nSaved: {PATIENT}_per_record.csv, {PATIENT}_false_alarms.csv, "
      f"appended row to {master}")

[SEIZ] chb01_03  params={'select__percentile': 5, 'svm__C': 0.1, 'svm__gamma': 0.01}  thr=0.605 k=6  test_pos=16
[SEIZ] chb01_04  params={'select__percentile': 5, 'svm__C': 0.1, 'svm__gamma': 0.01}  thr=0.621 k=4  test_pos=11
[SEIZ] chb01_15  params={'select__percentile': 10, 'svm__C': 0.1, 'svm__gamma': 'scale'}  thr=-0.113 k=8  test_pos=16
[SEIZ] chb01_16  params={'select__percentile': 15, 'svm__C': 0.1, 'svm__gamma': 'scale'}  thr=0.579 k=8  test_pos=21
[SEIZ] chb01_18  params={'select__percentile': 15, 'svm__C': 0.1, 'svm__gamma': 'scale'}  thr=0.530 k=8  test_pos=37
[SEIZ] chb01_21  params={'select__percentile': 5, 'svm__C': 0.1, 'svm__gamma': 0.01}  thr=0.506 k=6  test_pos=38
[SEIZ] chb01_26  params={'select__percentile': 5, 'svm__C': 0.1, 'svm__gamma': 0.01}  thr=0.463 k=6  test_pos=41
[NON-MODEL] params={'select__percentile': 5, 'svm__C': 0.1, 'svm__gamma': 0.01}  thr=0.633 k=6  (scores all seizure-free records)

===== SEGMENT-LEVEL (pooled OOF) =====
AUPRC   : 0.352
ROC-AUC : 

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (
    GroupKFold, GridSearchCV, LeaveOneGroupOut, cross_val_predict,
)
from sklearn.metrics import average_precision_score, roc_auc_score, make_scorer
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 8               # clinical cap on persistence (latency control)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 5       # a-priori ceiling (looser than Shoeb; GC trades FP for latency)
SELECT_FPR        = 0.01      # inner-CV selection: cap on segment-level FPR for the proxy scorer

OUTER_N_JOBS = 8              # parallelism over LORO folds (inner stays at 1)

PATIENT = os.environ.get("PATIENT", "chbXX")   # filename prefix; set per run e.g. PATIENT=chb01

# percentile (feature selection), C, and gamma are all tuned via the grid below
param_grid = {
    "select__percentile": [5, 10, 15],
    "svm__C": [0.1],
    "svm__gamma": ["scale"],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf", class_weight="balanced",
                    cache_size=8000)),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    pos = (sc >= thr).astype(int)
    alarm_end_times, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            alarm_end_times.append(i * STEP + WINDOW_LEN)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in alarm_end_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat

def _overlap(a, b):
    return a[0] < b[1] and b[0] < a[1]

def _safe_div(a, b):
    return a / b if b else float("nan")

# --- SEGMENT-LEVEL (window-level) confusion-matrix metrics at a given threshold ---
# Thresholds the raw window scores (no k-persistence; that is event-level logic).
# Called with thr=0.0 (the natural decision_function boundary) for standard,
# literature-comparable segment-level reporting.
# Returns accuracy, specificity, sensitivity, precision, f1.
# Undefined metrics (e.g. sensitivity on an all-negative record) -> NaN via _safe_div.
def segment_metrics(y, sc, thr):
    y = np.asarray(y).astype(int)
    pred = (sc >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    acc  = _safe_div(tp + tn, tp + tn + fp + fn)
    spec = _safe_div(tn, tn + fp)
    sens = _safe_div(tp, tp + fn)
    prec = _safe_div(tp, tp + fp)
    f1 = (_safe_div(2 * prec * sens, prec + sens)
          if (prec == prec and sens == sens) else float("nan"))
    return acc, spec, sens, prec, f1

# --- HYPERPARAMETER-SELECTION SCORER (inner CV) ---------------------------
# Objective: maximize segment-level sensitivity (recall) subject to a cap on
# the segment-level false-positive rate. This selects hyperparameters at the
# LOW-false-alarm operating point we actually deploy at, instead of averaging
# over the whole PR curve (which let high-C models win on irrelevant regions
# while flooding false positives at our operating point).
#
#   score = max over thresholds tau  of  Recall(tau)   s.t.  FPR(tau) <= alpha
#
# It is a SEGMENT-LEVEL SURROGATE for the event-level FP/day goal: it ignores
# k-persistence and duration normalization, but tracks the same low-FP regime.
# Returns 0 for single-class folds (undefined) -> worst score.
def recall_at_fpr(y_true, y_score, target_fpr=SELECT_FPR):
    y_true = np.asarray(y_true).astype(int)
    if y_true.sum() == 0 or y_true.sum() == len(y_true):
        return 0.0
    order = np.argsort(-y_score)
    ys = y_true[order]
    P = ys.sum(); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    fpr = fp / N; recall = tp / P
    feasible = recall[fpr <= target_fpr]
    return float(feasible.max()) if feasible.size else 0.0

# make_scorer API: sklearn < 1.4 uses needs_threshold=True; >= 1.4 uses
# response_method=. Detect by version rather than try/except, because 1.3
# silently accepts response_method and only fails later, at scoring time.
import sklearn
from packaging import version as _v
if _v.parse(sklearn.__version__) >= _v.parse("1.4"):
    EVENT_PROXY_SCORER = make_scorer(recall_at_fpr, response_method="decision_function")
else:
    EVENT_PROXY_SCORER = make_scorer(recall_at_fpr, needs_threshold=True)

def false_alarm_rows_of(record, s):
    # Event-level false alarms (hypothesis events not overlapping any reference event).
    ref_ev, hyp_ev = s.ref.events, s.hyp.events
    rows, aid = [], 0
    for hs, he in hyp_ev:
        if not any(_overlap((hs, he), r) for r in ref_ev):
            rows.append(dict(record=record, alarm_id=aid,
                             start_sec=float(hs), end_sec=float(he),
                             duration_sec=float(he - hs)))
            aid += 1
    return rows

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Within FP/day budget: max sensitivity, then LOWER k (latency), then F1, then low FP.
# If nothing meets budget: fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 25)))
    else:
        thr_grid = THRESH_GRID

    best = None
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=len(np.unique(groups)))
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring=EVENT_PROXY_SCORER, cv=inner, n_jobs=1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def n_selected_of(model):
    return int(model.named_steps["select"].get_support().sum())

def oof_train_record_scores(best_params, exclude):
    # Outer-training pool (all seizure records except the held-out test record)
    Xtr, ytr, grps = build_train(exclude=exclude)
    hp = {p: best_params[p] for p in ("select__percentile", "svm__C","svm__gamma")}
    est = make_pipe().set_params(**hp)

    # One LeaveOneGroupOut pass -> OOF decision scores for every training record
    oof_scores = cross_val_predict(
        est, Xtr, ytr, groups=grps,
        cv=LeaveOneGroupOut(), method="decision_function", n_jobs=1,
    )

    out = []
    for name in np.unique(grps):
        m = (grps == name)
        y_name = ytr[m]
        out.append((name, oof_scores[m], y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO  (outer folds parallelized; inner stays single-threaded)
# =====================================================
def process_seizure_fold(test_record):
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, _ = tune_fit(Xtr, ytr, grps)
    oof = oof_train_record_scores(best, exclude=test_record)
    thr, k = calibrate(oof)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, best=best, n_selected=n_selected_of(model))

results = Parallel(n_jobs=OUTER_N_JOBS)(
    delayed(process_seizure_fold)(r) for r in seizure_records.keys()
)

test_outputs = []
seg_scores, seg_labels = [], []
fold_choices = []
meta = {}   # per-record: type, C, gamma, selected_features
for res in results:  # Parallel preserves input order -> deterministic
    test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                         res["thr"], res["k"]))
    seg_scores.append(res["sc"]); seg_labels.append(res["y"])
    fold_choices.append((res["record"], res["thr"], res["k"]))
    meta[res["record"]] = dict(type="seizure", C=res["best"]["svm__C"],
                               gamma=res["best"]["svm__gamma"],
                               selected_features=res["n_selected"])
    print(f"[SEIZ] {res['record']}  params={res['best']}  "
          f"thr={res['thr']:.3f} k={res['k']}  test_pos={int(res['y'].sum())}")

# Non-model: one global threshold/k for all seizure-free records
Xall, yall, gall = build_train(exclude=None)
full_model, full_best, _ = tune_fit(Xall, yall, gall)
full_n_selected = n_selected_of(full_model)
oof_full = oof_train_record_scores(full_best, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
print(f"[NON-MODEL] params={full_best}  thr={thr_fp:.3f} k={k_fp}  (scores all seizure-free records)")
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)
    meta[name] = dict(type="nonseizure", C=full_best["svm__C"],
                      gamma=full_best["svm__gamma"],
                      selected_features=full_n_selected)

# =====================================================
# POOLED SEGMENT-LEVEL (all records combined) — threshold-free summary
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# PER-RECORD: segment-level (thresholded) + event-level + false-alarm log
# =====================================================
TP = FP = FN = 0
all_latencies = []
per_record_rows = []
false_alarm_rows = []
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    r_tp, r_fp, r_fn = s.tp, s.fp, s.refTrue - s.tp
    TP += r_tp; FP += r_fp; FN += r_fn

    # event-level metrics for this record
    ev_sens = _safe_div(r_tp, r_tp + r_fn)
    ev_prec = _safe_div(r_tp, r_tp + r_fp)
    ev_f1   = (_safe_div(2 * ev_sens * ev_prec, ev_sens + ev_prec)
               if (ev_sens == ev_sens and ev_prec == ev_prec) else float("nan"))

    lats = detection_latencies(record, sc, thr, k)
    all_latencies.extend(lats)

    # event-level false-alarm locations
    false_alarm_rows.extend(false_alarm_rows_of(record, s))

    # segment-level (window-level) metrics at the natural decision boundary
    # (decision_function >= 0), i.e. the classifier's own predict() boundary.
    # Kept independent of the event-level calibrated threshold so these numbers
    # are comparable to standard segment-level reporting in the literature.
    seg_acc, seg_spec, seg_sens, seg_prec, seg_f1 = segment_metrics(y, sc, 0.0)

    m = meta[record]
    per_record_rows.append({
        "patient": PATIENT,
        "record": record,
        "n_windows": int(len(y)),
        "n_seizure_windows": int(np.asarray(y).sum()),
        "record_length_sec": dur,
        # --- parameters ---
        "C": m["C"], "gamma": m["gamma"],
        "selected_features": m["selected_features"],
        "threshold": thr, "k": k,
        # --- segment-level (window-level, thresholded) ---
        "seg_accuracy": seg_acc,
        "seg_specificity": seg_spec,
        "seg_sensitivity": seg_sens,
        "seg_precision": seg_prec,
        "seg_f1": seg_f1,
        # --- event-level ---
        "event_sensitivity": ev_sens,
        "event_precision": ev_prec,
        "event_f1": ev_f1,
        "event_n_false_alarms": int(r_fp),
        "latency_median": float(np.median(lats)) if lats else np.nan,
    })

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

# =====================================================
# SAVE  (2 CSVs)
#   1) per-record metrics    2) event-level false-alarm locations
# =====================================================
per_record_cols = ["patient", "record", "n_windows", "n_seizure_windows",
                   "record_length_sec",
                   "C", "gamma", "selected_features", "threshold", "k",
                   "seg_accuracy", "seg_specificity", "seg_sensitivity",
                   "seg_precision", "seg_f1",
                   "event_sensitivity", "event_precision", "event_f1",
                   "event_n_false_alarms", "latency_median"]
pd.DataFrame(per_record_rows, columns=per_record_cols) \
  .to_csv(f"{PATIENT}_per_record.csv", index=False)

fa_cols = ["record", "alarm_id", "start_sec", "end_sec", "duration_sec"]
pd.DataFrame(false_alarm_rows, columns=fa_cols) \
  .to_csv(f"{PATIENT}_false_alarms.csv", index=False)

# Master summary -> APPEND one row per patient (builds the 24-patient table)
summary = {
    "patient": PATIENT,
    "auprc": average_precision_score(L, S),
    "roc_auc": roc_auc_score(L, S),
    "prev": L.mean(),
    "TP": TP, "FP": FP, "FN": FN,
    "sensitivity": sens, "precision": prec, "f1": f1,
    "FP/day": FP / days,
    "FP/hour": FP / (days * 24),
    "latency_median": med_lat,
    "n_detected": len(all_latencies),
}
master = "results_summary.csv"
pd.DataFrame([summary]).to_csv(
    master, mode="a", header=not os.path.exists(master), index=False,
)

print(f"\nSaved: {PATIENT}_per_record.csv, {PATIENT}_false_alarms.csv, "
      f"appended row to {master}")

: 

In [11]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (
    GroupKFold, GridSearchCV, LeaveOneGroupOut, cross_val_predict,
)
from sklearn.metrics import make_scorer
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 8               # clinical cap on persistence (latency control)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 5.0       # a-priori ceiling. NOTE: Shoeb is 0.08 FD/h ~= 1.9 FP/day;
                              # set this on a principled basis (e.g. Shoeb-matched), not to flatter results.
SELECT_FPR        = 0.01      # inner-CV selection: cap on segment-level FPR for the proxy scorer

OUTER_N_JOBS = 8              # parallelism over LORO folds (inner stays at 1)

PATIENT = os.environ.get("PATIENT", "chbXX")   # filename prefix; set per run e.g. PATIENT=chb01

# --- SzCORE event-scoring parameters, pinned explicitly (Dan et al., 2024) ---
SZCORE_PARAMS = scoring.EventScoring.Parameters(
    toleranceStart=30, toleranceEnd=60, minOverlap=0,
    maxEventDuration=5 * 60, minDurationBetweenEvents=90,
)

# percentile (feature selection), C, and gamma are all tuned via the grid below
param_grid = {
    "select__percentile": [5, 10, 15],
    "svm__C": [ 0.1],
    "svm__gamma": ["scale"],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf", class_weight="balanced",
                    cache_size=8000)),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

def _arming_times(sc, thr, k):
    # End time of the k-th consecutive positive window (the alarm-defining window),
    # one per run that reaches k. This is the latency convention used throughout.
    pos = (sc >= thr).astype(int)
    arm, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            arm.append(i * STEP + WINDOW_LEN)
    return arm

def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    arm_times = _arming_times(sc, thr, k)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in arm_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat

def _safe_div(a, b):
    return a / b if b else float("nan")

# --- SEGMENT-LEVEL (window-level) confusion-matrix metrics at a given threshold ---
# Thresholds the raw window scores (no k-persistence; that is event-level logic).
# Called with thr=0.0 (the natural decision_function boundary).
# Returns accuracy, specificity, sensitivity, precision, f1 and the raw tp/fp/fn counts.
def segment_metrics(y, sc, thr):
    y = np.asarray(y).astype(int)
    pred = (sc >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    acc  = _safe_div(tp + tn, tp + tn + fp + fn)
    spec = _safe_div(tn, tn + fp)
    sens = _safe_div(tp, tp + fn)
    prec = _safe_div(tp, tp + fp)
    f1 = (_safe_div(2 * prec * sens, prec + sens)
          if (prec == prec and sens == sens) else float("nan"))
    return acc, spec, sens, prec, f1, tp, fp, fn

# --- HYPERPARAMETER-SELECTION SCORER (inner CV) ---------------------------
# Maximize segment-level recall subject to a cap on segment-level FPR: selects
# hyperparameters at the low-false-alarm operating point we deploy at.
def recall_at_fpr(y_true, y_score, target_fpr=SELECT_FPR):
    y_true = np.asarray(y_true).astype(int)
    if y_true.sum() == 0 or y_true.sum() == len(y_true):
        return 0.0
    order = np.argsort(-y_score)
    ys = y_true[order]
    P = ys.sum(); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    fpr = fp / N; recall = tp / P
    feasible = recall[fpr <= target_fpr]
    return float(feasible.max()) if feasible.size else 0.0

import sklearn
from packaging import version as _v
if _v.parse(sklearn.__version__) >= _v.parse("1.4"):
    EVENT_PROXY_SCORER = make_scorer(recall_at_fpr, response_method="decision_function")
else:
    EVENT_PROXY_SCORER = make_scorer(recall_at_fpr, needs_threshold=True)

def alarm_rows_of(record, s, sc, thr, k):
    # Every alarm (hypothesis event), flagged true/false exactly as timescoring
    # counts s.tp / s.fp (overlap with tpMask = ref events, extended by 30/60s
    # tolerance, that were detected). For true alarms, latency = arming time
    # (end of k-th consecutive window) inside the alarm and inside the detected
    # seizure's [onset-30, offset+60] window, minus that seizure's onset.
    # #false-alarms == s.fp by construction.
    TOL_PRE, TOL_POST = 30, 60
    fs = s.fs
    arm_times = _arming_times(sc, thr, k)
    seiz = [(float(r["start_sec"]), float(r["end_sec"]))
            for _, r in SEIZ[SEIZ["file"] == record].iterrows()]
    rows, aid = [], 0
    for hs, he in s.hyp.events:
        a, b = round(hs * fs), round(he * fs)
        is_true = bool(np.any(s.tpMask[a:b]))
        latency = float("nan")
        if is_true:
            cands = []
            for onset, offset in seiz:
                if hs <= (offset + TOL_POST) and (onset - TOL_PRE) <= he:  # extended overlap
                    arms_in = [t for t in arm_times
                               if (onset - TOL_PRE) <= t <= (offset + TOL_POST)
                               and (hs - 1e-6) <= t <= (he + 1e-6)]
                    if arms_in:
                        cands.append(min(arms_in) - onset)
            if cands:
                latency = float(min(cands))
        rows.append(dict(record=record, alarm_id=aid,
                         start_sec=float(hs), end_sec=float(he),
                         duration_sec=float(he - hs),
                         alarm_type="true" if is_true else "false",
                         latency=latency))
        aid += 1
    return rows

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 25)))
    else:
        thr_grid = THRESH_GRID

    best = None
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=len(np.unique(groups)))
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring=EVENT_PROXY_SCORER, cv=inner, n_jobs=1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def n_selected_of(model):
    return int(model.named_steps["select"].get_support().sum())

def oof_train_record_scores(best_params, exclude):
    Xtr, ytr, grps = build_train(exclude=exclude)
    hp = {p: best_params[p] for p in ("select__percentile", "svm__C","svm__gamma")}
    est = make_pipe().set_params(**hp)
    oof_scores = cross_val_predict(
        est, Xtr, ytr, groups=grps,
        cv=LeaveOneGroupOut(), method="decision_function", n_jobs=1,
    )
    out = []
    for name in np.unique(grps):
        m = (grps == name)
        y_name = ytr[m]
        out.append((name, oof_scores[m], y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO  (outer folds parallelized; inner stays single-threaded)
# =====================================================
def process_seizure_fold(test_record):
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, _ = tune_fit(Xtr, ytr, grps)
    oof = oof_train_record_scores(best, exclude=test_record)
    thr, k = calibrate(oof)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, best=best, n_selected=n_selected_of(model))

results = Parallel(n_jobs=OUTER_N_JOBS)(
    delayed(process_seizure_fold)(r) for r in seizure_records.keys()
)

test_outputs = []
fold_choices = []
meta = {}
for res in results:
    test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                         res["thr"], res["k"]))
    fold_choices.append((res["record"], res["thr"], res["k"]))
    meta[res["record"]] = dict(type="seizure", C=res["best"]["svm__C"],
                               gamma=res["best"]["svm__gamma"],
                               selected_features=res["n_selected"])
    print(f"[SEIZ] {res['record']}  params={res['best']}  "
          f"thr={res['thr']:.3f} k={res['k']}  test_pos={int(res['y'].sum())}")

# Non-model: one global threshold/k for all seizure-free records
Xall, yall, gall = build_train(exclude=None)
full_model, full_best, _ = tune_fit(Xall, yall, gall)
full_n_selected = n_selected_of(full_model)
oof_full = oof_train_record_scores(full_best, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
print(f"[NON-MODEL] params={full_best}  thr={thr_fp:.3f} k={k_fp}  (scores all seizure-free records)")
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    meta[name] = dict(type="nonseizure", C=full_best["svm__C"],
                      gamma=full_best["svm__gamma"],
                      selected_features=full_n_selected)

# =====================================================
# PER-RECORD pass -> segment-level table + alarm log + event totals
# Segment ratio metrics are only meaningful on seizure records; on seizure-free
# records they are set to NaN (raw tp/fp/fn are still reported, since those are
# honest counts and the fp count is the false-alarm contribution).
# =====================================================
TP = FP = FN = 0
all_latencies = []
segment_rows = []
alarm_rows = []
for record, sc, y, dur, thr, k in test_outputs:
    is_seiz = meta[record]["type"] == "seizure"

    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS),
                             Annotation(hyp, GRID_FS), SZCORE_PARAMS)
    TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp   # non-seizure FPs count toward FP/day

    alarm_rows.extend(alarm_rows_of(record, s, sc, thr, k))
    if is_seiz:
        all_latencies.extend(detection_latencies(record, sc, thr, k))

    seg_acc, seg_spec, seg_sens, seg_prec, seg_f1, seg_tp, seg_fp, seg_fn = \
        segment_metrics(y, sc, 0.0)
    if not is_seiz:
        seg_acc = seg_spec = seg_sens = seg_prec = seg_f1 = np.nan

    m = meta[record]
    segment_rows.append({
        "patient": PATIENT,
        "record": record,
        "n_windows": int(len(y)),
        "n_seizures_windows": int(np.asarray(y).sum()),
        "record_length": dur,
        "C": m["C"], "gamma": m["gamma"],
        "selected_features": m["selected_features"],
        "threshold": thr, "k": k,
        "seg_accuracy": seg_acc,
        "seg_specificity": seg_spec,
        "seg_sensitivity": seg_sens,
        "seg_precision": seg_prec,
        "seg_f1": seg_f1,
        "tp": seg_tp, "fp": seg_fp, "fn": seg_fn,
    })

# event-level totals
sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
total_len_sec = sum(t[3] for t in test_outputs)
days = total_len_sec / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")

print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Total length  : {total_len_sec/3600:.2f} h")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, kk in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={kk}")

# =====================================================
# SAVE
#   1) SEGMENT_PER_RECORD_<PATIENT>.csv   segment-level metrics per record
#   2) <PATIENT>_ALARMS.csv               every alarm, true/false + latency
#   3) RESULTS_SUMMARY.csv                one appended row per patient
# =====================================================
segment_cols = ["patient", "record", "n_windows", "n_seizures_windows",
                "record_length", "C", "gamma", "selected_features",
                "threshold", "k",
                "seg_accuracy", "seg_specificity", "seg_sensitivity",
                "seg_precision", "seg_f1", "tp", "fp", "fn"]
pd.DataFrame(segment_rows, columns=segment_cols) \
  .to_csv(f"SEGMENT_PER_RECORD_{PATIENT}.csv", index=False)

alarm_cols = ["record", "alarm_id", "start_sec", "end_sec", "duration_sec",
              "alarm_type", "latency"]
pd.DataFrame(alarm_rows, columns=alarm_cols) \
  .to_csv(f"{PATIENT}_ALARMS.csv", index=False)

summary = {
    "patient": PATIENT,
    "total_length_sec": total_len_sec,
    "total_length_hours": total_len_sec / 3600,
    "TP": TP, "FP": FP, "FN": FN,
    "sensitivity": sens, "precision": prec, "f1": f1,
    "FP/day": FP / days,
    "FP/hour": FP / (days * 24),
    "latency_median": med_lat,
    "n_detected": len(all_latencies),
}
master = "RESULTS_SUMMARY.csv"
pd.DataFrame([summary]).to_csv(
    master, mode="a", header=not os.path.exists(master), index=False,
)

print(f"\nSaved: SEGMENT_PER_RECORD_{PATIENT}.csv, {PATIENT}_ALARMS.csv, "
      f"appended row to {master}")

: 

: 

: 